# Milestone 7 — Leakage-Safe SFT Training-Data Design

**Goal:** turn public benchmark examples into a small, reproducible SFT package that teaches the residual behaviors identified in Milestone 6—without contaminating the protected `financial` evaluation database.

This notebook is a **data-quality pipeline**, not a fine-tuning run. It answers four questions:

1. Which examples are legally and experimentally eligible?
2. Do their databases, metadata, and SQL actually work?
3. Are the labels leakage-safe, behaviorally useful, and semantically credible?
4. Can they be exported in the exact chat format the model will see at inference time?

> **Core principle:** SFT amplifies demonstrations. More records are not useful if they demonstrate the wrong behavior.


## Pipeline and quality gates

```text
source inventory
  → database/metadata validation
  → SQL parse + execution validation
  → reference-only compatibility normalization
  → coverage analysis
  → leakage + duplicate controls
  → semantic audit and curation
  → canonical chat records
  → provider-ready JSONL + manifest
```

Each gate is stricter than the previous one. A query that executes is technically usable, but it is not necessarily a correct answer to its question.


In [30]:
from collections import Counter, defaultdict
from hashlib import sha256
from pathlib import Path
from time import monotonic
import hashlib
import json
import math
import re
import sqlite3
import statistics
import unicodedata

import sqlglot
from sqlglot import exp, parse, parse_one

from domain_adapt.finch import make_reference_compatible
from domain_adapt.frontier import build_model_input


def find_repository_root(start_path: Path) -> Path:
    """Find the repository root from either the repo or notebooks directory."""
    start_path = start_path.resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not identify the repository root")


repository_root = find_repository_root(Path.cwd())
DATA_DIR = repository_root / "data"
DATASET_PATH = DATA_DIR / "finch_dataset.json"
MANIFEST_PATH = repository_root / "configs" / "eval_manifest_v1.json"
TRAINING_DIR = DATA_DIR / "training"
TRAINING_DATABASE_ROOT = DATA_DIR / "training_databases"
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

finch_records = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

print("Repository root:", repository_root)
print("FINCH records:", len(finch_records))


Repository root: C:\Users\shchitt\Downloads\Projects\domain-adapt
FINCH records: 75725


## 1. Protect evaluation before selecting training data

All 106 BIRD `financial` examples are protected—not only the 12 working questions and 24 frozen questions. They are excluded from training, validation, few-shot selection, synthetic prompting, and semantic repair. This prevents direct memorization and subtler schema leakage.


In [ ]:
scope = manifest["scope"]

financial_records = [
    record
    for record in finch_records
    if record["db_name"] == scope["db_name"]
    and record["db_id"] == scope["db_id"]
]

outside_financial = [
    record
    for record in finch_records
    if not (
        record["db_name"] == scope["db_name"]
        and record["db_id"] == scope["db_id"]
    )
]

working_ids = set(manifest["selection"]["working_dev"]["ids"])
frozen_ids = set(manifest["selection"]["frozen_eval"]["ids"])
financial_ids = {record["question_id"] for record in financial_records}

assert len(financial_records) == 106
assert working_ids.isdisjoint(frozen_ids)
assert working_ids <= financial_ids
assert frozen_ids <= financial_ids

source_counts = Counter(record["db_name"] for record in outside_financial)
source_database_counts = Counter(
    (record["db_name"], record["db_id"])
    for record in outside_financial
)
other_bird_counts = Counter(
    record["db_id"]
    for record in outside_financial
    if record["db_name"] == "bird"
)

print("Protected BIRD financial records:", len(financial_records))
print("Financial records eligible for training: 0")
print("Other BIRD databases:", dict(other_bird_counts))


### Observed source inventory

FINCH contains 75,725 records, but only six non-protected BIRD databases were considered. BIRD was preferred because it is closest to the target task and provides executable SQLite assets. Structural similarity is useful; shared evaluation examples are not.


In [ ]:
from collections import defaultdict
from sqlglot import parse_one, exp


BIRD_CANDIDATES = {
    "retails",
    "regional_sales",
    "sales",
    "sales_in_weather",
    "debit_card_specializing",
    "retail_world",
}


def profile_sql(sql: str) -> dict:
    try:
        tree = parse_one(sql, read="sqlite")
    except Exception:
        return {"parsed": False}

    tables = {
        table.name
        for table in tree.find_all(exp.Table)
    }

    select = tree.find(exp.Select)
    columns = list(tree.find_all(exp.Column))

    has_aggregation = any(
        isinstance(node, exp.AggFunc)
        for node in tree.walk()
    )
    has_case = any(
        isinstance(node, exp.Case)
        for node in tree.walk()
    )

    return {
        "parsed": True,
        "table_count": len(tables),
        "join_count": sum(
            1 for _ in tree.find_all(exp.Join)
        ),
        "projection_count": (
            len(select.expressions) if select else 0
        ),
        "has_aggregation": has_aggregation,
        "has_grouping": tree.find(exp.Group) is not None,
        "has_conditional_aggregation": (
            has_aggregation and has_case
        ),
        "has_arithmetic": any(
            isinstance(
                node,
                (exp.Add, exp.Sub, exp.Mul, exp.Div)
            )
            for node in tree.walk()
        ),
        "has_subquery": tree.find(exp.Subquery) is not None,
        "has_cte": tree.find(exp.With) is not None,
        "has_qualified_columns": any(
            bool(column.table)
            for column in columns
        ),
    }


profiles_by_database = defaultdict(list)

for record in outside_financial:
    if (
        record["db_name"] == "bird"
        and record["db_id"] in BIRD_CANDIDATES
    ):
        profiles_by_database[record["db_id"]].append(
            profile_sql(record["SQL"])
        )


def percentage(count: int, total: int) -> str:
    return f"{count / total:.0%}" if total else "n/a"


header = (
    f"{'Database':<27} {'N':>4} {'Parse':>7} "
    f"{'2+ tbl':>7} {'3+ tbl':>7} {'Agg':>7} "
    f"{'Cond':>7} {'Arith':>7} {'Subq':>7}"
)
print(header)
print("-" * len(header))

for database_id in sorted(profiles_by_database):
    profiles = profiles_by_database[database_id]
    total = len(profiles)

    def count(predicate):
        return sum(
            1 for profile in profiles
            if profile.get("parsed") and predicate(profile)
        )

    parsed = sum(
        profile.get("parsed", False)
        for profile in profiles
    )

    print(
        f"{database_id:<27} "
        f"{total:>4} "
        f"{percentage(parsed, total):>7} "
        f"{percentage(count(lambda p: p['table_count'] >= 2), total):>7} "
        f"{percentage(count(lambda p: p['table_count'] >= 3), total):>7} "
        f"{percentage(count(lambda p: p['has_aggregation']), total):>7} "
        f"{percentage(count(lambda p: p['has_conditional_aggregation']), total):>7} "
        f"{percentage(count(lambda p: p['has_arithmetic']), total):>7} "
        f"{percentage(count(lambda p: p['has_subquery']), total):>7}"
    )


## 2. Select schemas, not random rows

The split is database-disjoint so validation measures transfer to an unseen schema.

| Role | Requested database | Decision |
|---|---|---|
| Train | `debit_card_specializing` | Keep: finance-adjacent, strong joins and arithmetic |
| Train | `regional_sales` | Keep: strong subquery and conditional-aggregation coverage |
| Train | `retail_world` | Reject after asset validation: metadata does not match SQLite |
| Validation | `sales` | Keep as a fully held-out schema |

`retails` and `sales_in_weather` were deferred because their local assets were disproportionately large for this learning experiment. The aim is sufficient behavioral coverage, not maximum volume.

**Source limitation:** FINCH is CC-BY-NC-4.0. These assets remain local and are suitable here for research and education, not unrestricted commercial training.


## 3. Validate database assets before trusting labels

Required local layout:

```text
data/training_databases/
  tables.json
  debit_card_specializing/debit_card_specializing.sqlite
  regional_sales/regional_sales.sqlite
  retail_world/retail_world.sqlite
  sales/sales.sqlite
```

The next cell checks file integrity, SQLite health, source family, tables, and columns. `retail_world` is deliberately rejected because its metadata omits `Territories` and `Employees.HireDate`.


In [ ]:
from hashlib import sha256
from pathlib import Path
import json
import sqlite3


REQUESTED_TRAINING_DATABASE_IDS = (
    "debit_card_specializing",
    "regional_sales",
    "retail_world",
)

REQUESTED_VALIDATION_DATABASE_IDS = (
    "sales",
)

ALL_DATABASE_IDS = (
    *REQUESTED_TRAINING_DATABASE_IDS,
    *REQUESTED_VALIDATION_DATABASE_IDS,
)

TABLES_PATH = (
    TRAINING_DATABASE_ROOT / "tables.json"
)


def quote_identifier(identifier: str) -> str:
    """Safely quote a SQLite identifier."""
    return '"' + identifier.replace('"', '""') + '"'


# Load FINCH metadata.
if not TABLES_PATH.exists():
    raise FileNotFoundError(TABLES_PATH)

table_metadata = json.loads(
    TABLES_PATH.read_text(encoding="utf-8")
)

# FINCH tables.json stores each database name as a key,
# alongside a separate "database" source-family field.
metadata_by_database = {}
metadata_source_by_database = {}

for item in table_metadata:
    database_keys = [
        key
        for key in item
        if key != "database"
    ]

    if len(database_keys) != 1:
        raise ValueError(
            "Unexpected tables.json entry with keys: "
            f"{sorted(item.keys())}"
        )

    database_id = database_keys[0]

    if database_id in metadata_by_database:
        raise ValueError(
            f"Duplicate metadata entry: {database_id}"
        )

    metadata_by_database[database_id] = (
        item[database_id]
    )
    metadata_source_by_database[database_id] = (
        item.get("database")
    )


asset_inventory = {}

for database_id in ALL_DATABASE_IDS:
    split = (
        "train"
        if database_id
        in REQUESTED_TRAINING_DATABASE_IDS
        else "validation"
    )

    sqlite_path = (
        TRAINING_DATABASE_ROOT
        / database_id
        / f"{database_id}.sqlite"
    )

    issues = []
    tables = []
    quick_check = None
    database_sha256 = None
    size_bytes = None
    sqlite_columns_by_table = {}

    # Check that the SQLite file exists.
    if not sqlite_path.exists():
        issues.append(
            f"missing SQLite file: {sqlite_path}"
        )
    else:
        size_bytes = sqlite_path.stat().st_size
        database_sha256 = sha256(
            sqlite_path.read_bytes()
        ).hexdigest()

        connection = sqlite3.connect(
            f"file:{sqlite_path.as_posix()}?mode=ro",
            uri=True,
        )

        try:
            quick_check = connection.execute(
                "PRAGMA quick_check"
            ).fetchone()[0]

            if quick_check != "ok":
                issues.append(
                    f"PRAGMA quick_check returned "
                    f"{quick_check!r}"
                )

            tables = [
                row[0]
                for row in connection.execute(
                    """
                    SELECT name
                    FROM sqlite_master
                    WHERE type = 'table'
                      AND name NOT LIKE 'sqlite_%'
                    ORDER BY name
                    """
                )
            ]

            if not tables:
                issues.append(
                    "SQLite database contains no user tables"
                )

            for table_name in tables:
                quoted_table = quote_identifier(
                    table_name
                )

                sqlite_columns_by_table[
                    table_name.casefold()
                ] = {
                    row[1].casefold(): row[1]
                    for row in connection.execute(
                        f"PRAGMA table_info({quoted_table})"
                    )
                }

        finally:
            connection.close()

    # Check that metadata exists.
    metadata = metadata_by_database.get(
        database_id
    )

    if metadata is None:
        issues.append(
            "missing tables.json metadata"
        )
        metadata = {}

    source_family = metadata_source_by_database.get(
        database_id
    )

    if source_family != "bird":
        issues.append(
            "unexpected metadata source family: "
            f"{source_family!r}"
        )

    # Compare SQLite and metadata tables.
    sqlite_table_names = {
        table.casefold(): table
        for table in tables
    }

    metadata_table_names = {
        table.casefold(): table
        for table in metadata.keys()
    }

    missing_from_metadata = sorted(
        sqlite_table_names[key]
        for key in (
            sqlite_table_names.keys()
            - metadata_table_names.keys()
        )
    )

    missing_from_sqlite = sorted(
        metadata_table_names[key]
        for key in (
            metadata_table_names.keys()
            - sqlite_table_names.keys()
        )
    )

    if missing_from_metadata:
        issues.append(
            "tables missing from metadata: "
            f"{missing_from_metadata}"
        )

    if missing_from_sqlite:
        issues.append(
            "metadata tables missing from SQLite: "
            f"{missing_from_sqlite}"
        )

    # Compare columns for tables present in both sources.
    column_mismatches = {}

    for folded_table_name in sorted(
        sqlite_table_names.keys()
        & metadata_table_names.keys()
    ):
        sqlite_table_name = sqlite_table_names[
            folded_table_name
        ]
        metadata_table_name = metadata_table_names[
            folded_table_name
        ]

        sqlite_columns = sqlite_columns_by_table.get(
            folded_table_name,
            {},
        )

        metadata_columns = {
            column["column_name"].casefold():
            column["column_name"]
            for column in metadata[
                metadata_table_name
            ].get("columns_info", [])
        }

        columns_missing_from_metadata = sorted(
            sqlite_columns[key]
            for key in (
                sqlite_columns.keys()
                - metadata_columns.keys()
            )
        )

        columns_missing_from_sqlite = sorted(
            metadata_columns[key]
            for key in (
                metadata_columns.keys()
                - sqlite_columns.keys()
            )
        )

        if (
            columns_missing_from_metadata
            or columns_missing_from_sqlite
        ):
            column_mismatches[
                sqlite_table_name
            ] = {
                "missing_from_metadata":
                    columns_missing_from_metadata,
                "missing_from_sqlite":
                    columns_missing_from_sqlite,
            }

    if column_mismatches:
        issues.append(
            "column mismatches detected"
        )

    status = (
        "accepted_for_query_validation"
        if not issues
        else "rejected_at_asset_validation"
    )

    asset_inventory[database_id] = {
        "split": split,
        "status": status,
        "sqlite_path": str(sqlite_path),
        "database_sha256": database_sha256,
        "size_bytes": size_bytes,
        "quick_check": quick_check,
        "tables": tables,
        "missing_from_metadata":
            missing_from_metadata,
        "missing_from_sqlite":
            missing_from_sqlite,
        "column_mismatches":
            column_mismatches,
        "issues": issues,
    }

    print("=" * 72)
    print("Database:", database_id)
    print("Requested split:", split)
    print("Status:", status)
    print("Quick check:", quick_check)
    print("Size bytes:", size_bytes)
    print("Tables:", tables)
    print("SHA-256:", database_sha256)

    if issues:
        print("Issues:")
        for issue in issues:
            print(" -", issue)

        if missing_from_metadata:
            print(
                "Missing from metadata:",
                missing_from_metadata,
            )

        if missing_from_sqlite:
            print(
                "Missing from SQLite:",
                missing_from_sqlite,
            )

        if column_mismatches:
            print(
                "Column mismatches:",
                json.dumps(
                    column_mismatches,
                    indent=2,
                ),
            )


approved_training_database_ids = tuple(
    database_id
    for database_id
    in REQUESTED_TRAINING_DATABASE_IDS
    if asset_inventory[database_id]["status"]
    == "accepted_for_query_validation"
)

approved_validation_database_ids = tuple(
    database_id
    for database_id
    in REQUESTED_VALIDATION_DATABASE_IDS
    if asset_inventory[database_id]["status"]
    == "accepted_for_query_validation"
)

rejected_database_ids = tuple(
    database_id
    for database_id in ALL_DATABASE_IDS
    if asset_inventory[database_id]["status"]
    == "rejected_at_asset_validation"
)

print()
print("=" * 72)
print("Asset-validation summary")
print(
    "Approved training databases:",
    approved_training_database_ids,
)
print(
    "Approved validation databases:",
    approved_validation_database_ids,
)
print(
    "Rejected databases:",
    rejected_database_ids,
)

if not approved_training_database_ids:
    raise RuntimeError(
        "No training databases passed asset validation."
    )

if not approved_validation_database_ids:
    raise RuntimeError(
        "No validation database passed asset validation."
    )


**Asset result:** 228 training candidates and 107 validation candidates remain across three approved databases. Asset rejection is a quality feature: incomplete schema context can make a correct reference query useless as an SFT demonstration.


In [ ]:
from collections import Counter
from time import monotonic
import sqlite3

from sqlglot import parse, exp


QUERY_TIMEOUT_SECONDS = 15
LARGE_RESULT_THRESHOLD = 10_000


training_candidates = [
    record
    for record in finch_records
    if record["db_name"] == "bird"
    and record["db_id"]
    in approved_training_database_ids
]

validation_candidates = [
    record
    for record in finch_records
    if record["db_name"] == "bird"
    and record["db_id"]
    in approved_validation_database_ids
]

candidate_records = (
    training_candidates
    + validation_candidates
)

assert len(training_candidates) == 228
assert len(validation_candidates) == 107
assert len(candidate_records) == 335


## 4. Parse and execute every reference

Technical validation checks that every target is a single read-only SQLite statement, executes within 15 seconds, and does not produce an obviously suspicious empty, null, zero-only, or very large result.

This is the expensive step (about six minutes on the observed machine). Its JSON artifact lets later work be audited without rerunning the model or exposing protected answers.


In [ ]:
def validate_sql_structure(sql: str) -> dict:
    """Parse and verify a single read-only SQL statement."""
    try:
        statements = [
            statement
            for statement in parse(
                sql,
                read="sqlite",
            )
            if statement is not None
        ]
    except Exception as error:
        return {
            "parsed": False,
            "single_statement": False,
            "read_only": False,
            "parse_error": (
                f"{type(error).__name__}: {error}"
            ),
        }

    if len(statements) != 1:
        return {
            "parsed": True,
            "single_statement": False,
            "read_only": False,
            "parse_error": None,
        }

    tree = statements[0]

    forbidden_node_names = {
        "Insert",
        "Update",
        "Delete",
        "Create",
        "Drop",
        "Alter",
        "Merge",
        "Command",
    }

    contains_forbidden_operation = any(
        type(node).__name__
        in forbidden_node_names
        for node in tree.walk()
    )

    has_select = (
        tree.find(exp.Select) is not None
    )

    return {
        "parsed": True,
        "single_statement": True,
        "read_only": (
            has_select
            and not contains_forbidden_operation
        ),
        "parse_error": None,
    }


def execute_with_timeout(
    connection: sqlite3.Connection,
    sql: str,
) -> dict:
    """Execute SQL fully without storing its complete result."""
    deadline = (
        monotonic() + QUERY_TIMEOUT_SECONDS
    )

    def should_interrupt() -> int:
        return int(monotonic() >= deadline)

    connection.set_progress_handler(
        should_interrupt,
        10_000,
    )

    row_count = 0
    column_count = None
    contains_null = False
    first_row = None

    try:
        cursor = connection.execute(sql)

        column_count = len(
            cursor.description or ()
        )

        for row in cursor:
            if first_row is None:
                first_row = row

            row_count += 1

            if any(value is None for value in row):
                contains_null = True

        return {
            "executes": True,
            "execution_error": None,
            "timed_out": False,
            "row_count": row_count,
            "column_count": column_count,
            "contains_null": contains_null,
            "first_row": first_row,
        }

    except sqlite3.Error as error:
        message = str(error)

        return {
            "executes": False,
            "execution_error": (
                f"{type(error).__name__}: "
                f"{message}"
            ),
            "timed_out": (
                "interrupted" in message.casefold()
            ),
            "row_count": None,
            "column_count": None,
            "contains_null": None,
            "first_row": None,
        }

    finally:
        connection.set_progress_handler(
            None,
            0,
        )



TECHNICAL_VALIDATION_PATH = (
    DATA_DIR / "training" / "technical_validation_v1.json"
)

reuse_technical_artifact = False
if TECHNICAL_VALIDATION_PATH.exists():
    cached_technical = json.loads(
        TECHNICAL_VALIDATION_PATH.read_text(encoding="utf-8")
    )
    reuse_technical_artifact = (
        cached_technical.get("dataset_sha256")
        == manifest["source"]["dataset_sha256"]
        and cached_technical.get("approved_training_databases")
        == list(approved_training_database_ids)
        and cached_technical.get("approved_validation_databases")
        == list(approved_validation_database_ids)
    )

if reuse_technical_artifact:
    technical_validation = cached_technical["results"]
    technical_validation_sha256 = sha256(
        TECHNICAL_VALIDATION_PATH.read_bytes()
    ).hexdigest()
    print("Loaded cached technical validation:", TECHNICAL_VALIDATION_PATH)
    print("Candidate records:", len(technical_validation))
    print("Status counts:", dict(Counter(
        result["status"] for result in technical_validation
    )))
else:
    connections = {}

    for database_id in (
        *approved_training_database_ids,
        *approved_validation_database_ids,
    ):
        sqlite_path = Path(
            asset_inventory[database_id][
                "sqlite_path"
            ]
        )

        connections[database_id] = (
            sqlite3.connect(
                f"file:{sqlite_path.as_posix()}?mode=ro",
                uri=True,
            )
        )


    technical_validation = []

    try:
        for record in candidate_records:
            database_id = record["db_id"]
            sql = record["SQL"]

            structure = validate_sql_structure(
                sql
            )

            execution = {
                "executes": False,
                "execution_error": None,
                "timed_out": False,
                "row_count": None,
                "column_count": None,
                "contains_null": None,
                "first_row": None,
            }

            if (
                structure["parsed"]
                and structure["single_statement"]
                and structure["read_only"]
            ):
                execution = execute_with_timeout(
                    connections[database_id],
                    sql,
                )

            concerns = []

            if not structure["parsed"]:
                concerns.append("parse_failure")
            elif not structure[
                "single_statement"
            ]:
                concerns.append(
                    "multiple_statements"
                )
            elif not structure["read_only"]:
                concerns.append(
                    "not_single_read_only_query"
                )
            elif not execution["executes"]:
                concerns.append(
                    "execution_timeout"
                    if execution["timed_out"]
                    else "execution_failure"
                )
            else:
                if execution["row_count"] == 0:
                    concerns.append("empty_result")

                if execution["contains_null"]:
                    concerns.append(
                        "contains_null"
                    )

                first_row = execution["first_row"]

                if (
                    execution["row_count"] == 1
                    and execution["column_count"] == 1
                    and first_row is not None
                    and isinstance(
                        first_row[0],
                        (int, float),
                    )
                    and first_row[0] == 0
                ):
                    concerns.append(
                        "single_numeric_zero"
                    )

                if (
                    execution["row_count"]
                    is not None
                    and execution["row_count"]
                    > LARGE_RESULT_THRESHOLD
                ):
                    concerns.append(
                        "very_large_result"
                    )

            if not structure["parsed"]:
                status = "rejected_technical"
            elif not structure[
                "single_statement"
            ]:
                status = "rejected_technical"
            elif not structure["read_only"]:
                status = "rejected_technical"
            elif not execution["executes"]:
                status = "rejected_technical"
            elif concerns:
                status = "needs_manual_review"
            else:
                status = (
                    "passed_technical_validation"
                )

            split = (
                "train"
                if database_id
                in approved_training_database_ids
                else "validation"
            )

            technical_validation.append({
                "question_id":
                    record["question_id"],
                "database_id": database_id,
                "split": split,
                "difficulty":
                    record["difficulty"],
                "status": status,
                "concerns": concerns,
                "parsed":
                    structure["parsed"],
                "single_statement":
                    structure[
                        "single_statement"
                    ],
                "read_only":
                    structure["read_only"],
                "parse_error":
                    structure["parse_error"],
                "executes":
                    execution["executes"],
                "execution_error":
                    execution[
                        "execution_error"
                    ],
                "timed_out":
                    execution["timed_out"],
                "row_count":
                    execution["row_count"],
                "column_count":
                    execution["column_count"],
                "contains_null":
                    execution["contains_null"],
            })

    finally:
        for connection in connections.values():
            connection.close()


    print("Candidate records:", len(
        technical_validation
    ))

    print(
        "Status counts:",
        dict(Counter(
            result["status"]
            for result in technical_validation
        )),
    )

    print(
        "Concern counts:",
        dict(Counter(
            concern
            for result in technical_validation
            for concern in result["concerns"]
        )),
    )

    print()
    print("By database:")

    for database_id in (
        *approved_training_database_ids,
        *approved_validation_database_ids,
    ):
        subset = [
            result
            for result in technical_validation
            if result["database_id"]
            == database_id
        ]

        print(
            database_id,
            dict(Counter(
                result["status"]
                for result in subset
            )),
        )


    # Save the technical validation results to a JSON file and compute its SHA-256 hash.

    from hashlib import sha256
    from pathlib import Path
    import json


    TECHNICAL_VALIDATION_PATH = (
        DATA_DIR
        / "training"
        / "technical_validation_v1.json"
    )

    TECHNICAL_VALIDATION_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    technical_validation_artifact = {
        "artifact_version": 1,
        "dataset_sha256": manifest[
            "source"
        ]["dataset_sha256"],
        "query_timeout_seconds":
            QUERY_TIMEOUT_SECONDS,
        "large_result_threshold":
            LARGE_RESULT_THRESHOLD,
        "approved_training_databases": list(
            approved_training_database_ids
        ),
        "approved_validation_databases": list(
            approved_validation_database_ids
        ),
        "rejected_asset_databases": list(
            rejected_database_ids
        ),
        "asset_inventory": asset_inventory,
        "results": technical_validation,
    }

    TECHNICAL_VALIDATION_PATH.write_text(
        json.dumps(
            technical_validation_artifact,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    technical_validation_sha256 = sha256(
        TECHNICAL_VALIDATION_PATH.read_bytes()
    ).hexdigest()

    print("Saved:", TECHNICAL_VALIDATION_PATH)
    print(
        "Technical-validation SHA-256:",
        technical_validation_sha256,
    )


### Initial technical result

Of 335 candidates, only 134 passed immediately; 194 required review and 7 timed out. Most suspicious empty, null, and zero results were not bad questions—they were caused by lower-cased reference literals being compared against mixed-case SQLite values.

This demonstrates a reusable rule: **investigate systematic data/runtime incompatibilities before discarding examples or blaming the model.**


## 5. Normalize references without weakening evaluation

Two narrow rules are applied only to trusted reference/training SQL:

- four-digit SQLite year extraction;
- case-insensitive equality for textual literals.

The evaluator never applies these repairs to model-generated SQL. Otherwise the harness would silently improve the candidate and inflate measured accuracy.


In [ ]:
from collections import Counter
from hashlib import sha256
from pathlib import Path
import json
import sqlite3

from sqlglot import parse_one, exp

from domain_adapt.finch import (
    make_reference_compatible,
)


def add_generalized_nocase(
    sql: str,
) -> tuple[str, bool]:
    """Add NOCASE to column/string equality expressions."""
    tree = parse_one(sql, read="sqlite")
    changed = False

    def transform(node):
        nonlocal changed

        if not isinstance(node, exp.EQ):
            return node

        left = node.this
        right = node.expression

        # column = 'literal'
        if (
            isinstance(left, exp.Column)
            and isinstance(right, exp.Literal)
            and right.is_string
        ):
            updated = node.copy()
            updated.set(
                "expression",
                exp.Collate(
                    this=right.copy(),
                    expression=exp.Var(
                        this="NOCASE"
                    ),
                ),
            )
            changed = True
            return updated

        # 'literal' = column
        if (
            isinstance(left, exp.Literal)
            and left.is_string
            and isinstance(right, exp.Column)
        ):
            updated = node.copy()
            updated.set(
                "this",
                exp.Collate(
                    this=left.copy(),
                    expression=exp.Var(
                        this="NOCASE"
                    ),
                ),
            )
            changed = True
            return updated

        return node

    transformed = tree.transform(transform)

    if not changed:
        return sql, False

    return (
        transformed.sql(dialect="sqlite"),
        True,
    )


def make_training_reference_compatible(
    sql: str,
) -> tuple[str, list[str]]:
    """
    Normalize known FINCH reference incompatibilities.

    Use only for reference/training SQL, never for
    model-generated evaluation SQL.
    """
    compatible_sql, rules = (
        make_reference_compatible(sql)
    )

    compatible_sql, generalized_changed = (
        add_generalized_nocase(
            compatible_sql
        )
    )

    if (
        generalized_changed
        and "case_insensitive_text_equality"
        not in rules
    ):
        rules.append(
            "case_insensitive_text_equality"
        )

    return compatible_sql, rules


def classify_execution(
    structure: dict,
    execution: dict,
) -> tuple[str, list[str]]:
    concerns = []

    if not structure["parsed"]:
        concerns.append("parse_failure")
    elif not structure["single_statement"]:
        concerns.append(
            "multiple_statements"
        )
    elif not structure["read_only"]:
        concerns.append(
            "not_single_read_only_query"
        )
    elif not execution["executes"]:
        concerns.append(
            "execution_timeout"
            if execution["timed_out"]
            else "execution_failure"
        )
    else:
        if execution["row_count"] == 0:
            concerns.append("empty_result")

        if execution["contains_null"]:
            concerns.append("contains_null")

        first_row = execution["first_row"]

        if (
            execution["row_count"] == 1
            and execution["column_count"] == 1
            and first_row is not None
            and isinstance(
                first_row[0],
                (int, float),
            )
            and first_row[0] == 0
        ):
            concerns.append(
                "single_numeric_zero"
            )

        if (
            execution["row_count"]
            is not None
            and execution["row_count"]
            > LARGE_RESULT_THRESHOLD
        ):
            concerns.append(
                "very_large_result"
            )

    if (
        not structure["parsed"]
        or not structure["single_statement"]
        or not structure["read_only"]
        or not execution["executes"]
    ):
        status = "rejected_technical"
    elif concerns:
        status = "needs_manual_review"
    else:
        status = (
            "passed_technical_validation"
        )

    return status, concerns



NORMALIZED_VALIDATION_PATH = (
    DATA_DIR / "training" / "reference_validation_v2.json"
)

reuse_normalized_artifact = False
if NORMALIZED_VALIDATION_PATH.exists():
    cached_normalized = json.loads(
        NORMALIZED_VALIDATION_PATH.read_text(encoding="utf-8")
    )
    reuse_normalized_artifact = (
        cached_normalized.get("parent_sha256")
        == technical_validation_sha256
        and cached_normalized.get("reference_only") is True
    )

if reuse_normalized_artifact:
    normalized_validation = cached_normalized["results"]
    normalized_validation_sha256 = sha256(
        NORMALIZED_VALIDATION_PATH.read_bytes()
    ).hexdigest()
    print("Loaded cached normalized validation:", NORMALIZED_VALIDATION_PATH)
    print("Records:", len(normalized_validation))
    print("Status counts:", dict(Counter(
        result["status"] for result in normalized_validation
    )))
else:
    original_validation_by_key = {
        (
            result["database_id"],
            result["question_id"],
        ): result
        for result in technical_validation
    }

    connections = {}

    for database_id in (
        *approved_training_database_ids,
        *approved_validation_database_ids,
    ):
        sqlite_path = Path(
            asset_inventory[database_id][
                "sqlite_path"
            ]
        )

        connections[database_id] = (
            sqlite3.connect(
                f"file:{sqlite_path.as_posix()}?mode=ro",
                uri=True,
            )
        )


    normalized_validation = []
    affected_count = 0
    revalidated_count = 0

    try:
        for record in candidate_records:
            key = (
                record["db_id"],
                record["question_id"],
            )

            previous = (
                original_validation_by_key[key]
            )

            original_sql = record["SQL"]

            compatible_sql, applied_rules = (
                make_training_reference_compatible(
                    original_sql
                )
            )

            changed = (
                compatible_sql != original_sql
            )

            if changed:
                affected_count += 1

            # Preserve previously rejected records. In particular,
            # do not spend another 15 seconds on known timeouts.
            if previous["status"] == (
                "rejected_technical"
            ):
                normalized_validation.append({
                    **previous,
                    "original_sql": original_sql,
                    "compatible_sql":
                        compatible_sql,
                    "compatibility_rules":
                        applied_rules,
                    "reference_changed":
                        changed,
                    "revalidated": False,
                })
                continue

            # Reuse the cached result if no SQL changed.
            if not changed:
                normalized_validation.append({
                    **previous,
                    "original_sql": original_sql,
                    "compatible_sql":
                        compatible_sql,
                    "compatibility_rules":
                        applied_rules,
                    "reference_changed":
                        False,
                    "revalidated": False,
                })
                continue

            revalidated_count += 1

            structure = validate_sql_structure(
                compatible_sql
            )

            execution = {
                "executes": False,
                "execution_error": None,
                "timed_out": False,
                "row_count": None,
                "column_count": None,
                "contains_null": None,
                "first_row": None,
            }

            if (
                structure["parsed"]
                and structure[
                    "single_statement"
                ]
                and structure["read_only"]
            ):
                execution = execute_with_timeout(
                    connections[
                        record["db_id"]
                    ],
                    compatible_sql,
                )

            status, concerns = (
                classify_execution(
                    structure,
                    execution,
                )
            )

            normalized_validation.append({
                "question_id":
                    record["question_id"],
                "database_id":
                    record["db_id"],
                "split": previous["split"],
                "difficulty":
                    record["difficulty"],
                "status": status,
                "concerns": concerns,
                "parsed":
                    structure["parsed"],
                "single_statement":
                    structure[
                        "single_statement"
                    ],
                "read_only":
                    structure["read_only"],
                "parse_error":
                    structure["parse_error"],
                "executes":
                    execution["executes"],
                "execution_error":
                    execution[
                        "execution_error"
                    ],
                "timed_out":
                    execution["timed_out"],
                "row_count":
                    execution["row_count"],
                "column_count":
                    execution["column_count"],
                "contains_null":
                    execution["contains_null"],
                "original_sql": original_sql,
                "compatible_sql":
                    compatible_sql,
                "compatibility_rules":
                    applied_rules,
                "reference_changed": True,
                "revalidated": True,
            })

    finally:
        for connection in connections.values():
            connection.close()


    original_status_by_key = {
        (
            result["database_id"],
            result["question_id"],
        ): result["status"]
        for result in technical_validation
    }

    recovered_count = sum(
        original_status_by_key[
            (
                result["database_id"],
                result["question_id"],
            )
        ] == "needs_manual_review"
        and result["status"]
        == "passed_technical_validation"
        for result in normalized_validation
    )

    regression_count = sum(
        original_status_by_key[
            (
                result["database_id"],
                result["question_id"],
            )
        ] == "passed_technical_validation"
        and result["status"]
        != "passed_technical_validation"
        for result in normalized_validation
    )

    print("Candidate records:", len(
        normalized_validation
    ))
    print("References changed:", affected_count)
    print("Queries revalidated:", revalidated_count)
    print(
        "Recovered from manual review:",
        recovered_count,
    )
    print(
        "Technical-pass regressions:",
        regression_count,
    )
    print(
        "Updated status counts:",
        dict(Counter(
            result["status"]
            for result in normalized_validation
        )),
    )
    print(
        "Remaining concern counts:",
        dict(Counter(
            concern
            for result in normalized_validation
            for concern in result["concerns"]
        )),
    )

    NORMALIZED_VALIDATION_PATH = (
        DATA_DIR
        / "training"
        / "reference_validation_v2.json"
    )

    normalized_artifact = {
        "artifact_version": 2,
        "parent_sha256":
            technical_validation_sha256,
        "compatibility_policy": [
            "sqlite_four_digit_year",
            "case_insensitive_text_equality",
        ],
        "reference_only": True,
        "results": normalized_validation,
    }

    NORMALIZED_VALIDATION_PATH.write_text(
        json.dumps(
            normalized_artifact,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    normalized_validation_sha256 = sha256(
        NORMALIZED_VALIDATION_PATH.read_bytes()
    ).hexdigest()

    print("Saved:", NORMALIZED_VALIDATION_PATH)
    print(
        "Normalized-validation SHA-256:",
        normalized_validation_sha256,
    )


### Normalization result

- 214 references changed and were revalidated.
- 176 records recovered from manual review.
- 310 records passed technical validation.
- 18 suspicious results and 7 technical failures remained excluded.

Original SQL, compatible SQL, applied rules, and execution evidence are retained separately for provenance.


## 6. Confirm the curriculum covers the hypothesis

Coverage is measured from observable SQL structure—not subjective labels. The important training behaviors are multi-table joins, qualification, exact projection, aggregation, arithmetic, conditional aggregation, and subqueries.


In [ ]:
clean_validation_results = [
    result
    for result in normalized_validation
    if result["status"]
    == "passed_technical_validation"
]

clean_candidate_keys = {
    (
        result["database_id"],
        result["question_id"],
    )
    for result in clean_validation_results
}

clean_records = [
    record
    for record in candidate_records
    if (
        record["db_id"],
        record["question_id"],
    ) in clean_candidate_keys
]

assert len(clean_records) == 310


coverage_records = []

for result in clean_validation_results:
    sql_profile = profile_sql(
        result["compatible_sql"]
    )

    assert sql_profile["parsed"]

    observable_tags = []

    if sql_profile["table_count"] >= 2:
        observable_tags.append(
            "multi_table"
        )

    if sql_profile["table_count"] >= 3:
        observable_tags.append(
            "three_plus_tables"
        )

    if sql_profile["join_count"] >= 1:
        observable_tags.append(
            "explicit_join"
        )

    if sql_profile[
        "has_qualified_columns"
    ]:
        observable_tags.append(
            "qualified_columns"
        )

    if sql_profile[
        "projection_count"
    ] >= 2:
        observable_tags.append(
            "multi_column_projection"
        )

    if sql_profile["has_aggregation"]:
        observable_tags.append(
            "aggregation"
        )

    if sql_profile[
        "has_conditional_aggregation"
    ]:
        observable_tags.append(
            "conditional_aggregation"
        )

    if sql_profile["has_arithmetic"]:
        observable_tags.append(
            "arithmetic"
        )

    if sql_profile["has_subquery"]:
        observable_tags.append(
            "subquery"
        )

    if sql_profile["has_cte"]:
        observable_tags.append("cte")

    coverage_records.append({
        "database_id":
            result["database_id"],
        "question_id":
            result["question_id"],
        "split": result["split"],
        "difficulty":
            result["difficulty"],
        "observable_tags":
            observable_tags,
        "table_count":
            sql_profile["table_count"],
        "join_count":
            sql_profile["join_count"],
        "projection_count":
            sql_profile[
                "projection_count"
            ],
    })


print(
    "Clean candidates by split:",
    dict(Counter(
        row["split"]
        for row in coverage_records
    )),
)

print()
print("Clean candidates by database:")

for database_id in (
    *approved_training_database_ids,
    *approved_validation_database_ids,
):
    subset = [
        row
        for row in coverage_records
        if row["database_id"]
        == database_id
    ]

    print(
        database_id,
        len(subset),
        dict(Counter(
            row["difficulty"]
            for row in subset
        )),
    )


for split in ("train", "validation"):
    subset = [
        row
        for row in coverage_records
        if row["split"] == split
    ]

    tag_counts = Counter(
        tag
        for row in subset
        for tag in row["observable_tags"]
    )

    print()
    print(
        f"{split.upper()} COVERAGE "
        f"({len(subset)} records)"
    )

    for tag, count in sorted(
        tag_counts.items()
    ):
        print(
            f"{tag:<28} "
            f"{count:>4} "
            f"({count / len(subset):>6.1%})"
        )


### Coverage result

The 225 technically clean training candidates contained strong join (76%), aggregation (56%), arithmetic (24%), subquery (24%), and conditional-aggregation (17%) coverage. The held-out `sales` schema supplied 85 clean validation candidates.

`regional_sales` dominates the training pool, so conclusions must remain scoped to this research experiment. More data was not added merely to balance counts; questionable examples would reduce label quality.


## 7. Prove separation and remove true duplicates

Leakage checks use normalized question hashes, canonical SQL hashes, near-question token similarity, and structural SQL fingerprints. SQL-shape overlap is expected—it is the transferable behavior SFT should learn. Protected or cross-split content overlap is not.


In [ ]:
from collections import defaultdict
from hashlib import sha256
import re
import unicodedata

from sqlglot import parse_one, exp


NEAR_DUPLICATE_THRESHOLD = 0.85


def normalize_question(text: str) -> str:
    text = unicodedata.normalize(
        "NFKC",
        text,
    ).casefold()

    tokens = re.findall(
        r"[a-z0-9_]+",
        text,
    )

    return " ".join(tokens)


def question_token_hashes(
    text: str,
) -> frozenset[str]:
    normalized = normalize_question(text)

    return frozenset(
        sha256(
            token.encode("utf-8")
        ).hexdigest()
        for token in normalized.split()
    )


def canonical_sql(sql: str) -> str:
    tree = parse_one(sql, read="sqlite")

    return tree.sql(
        dialect="sqlite",
        normalize=True,
        pretty=False,
    )


def sql_shape_hash(sql: str) -> str:
    """
    Hash SQL structure while ignoring literal values
    and identifier names.
    """
    tree = parse_one(sql, read="sqlite")

    node_types = [
        type(node).__name__
        for node in tree.walk()
        if not isinstance(
            node,
            (exp.Identifier, exp.Literal),
        )
    ]

    return sha256(
        "|".join(node_types).encode("utf-8")
    ).hexdigest()


def exact_hash(value: str) -> str:
    return sha256(
        value.encode("utf-8")
    ).hexdigest()


def jaccard_similarity(
    left: frozenset[str],
    right: frozenset[str],
) -> float:
    union = left | right

    if not union:
        return 1.0

    return len(left & right) / len(union)


normalized_result_by_key = {
    (
        result["database_id"],
        result["question_id"],
    ): result
    for result in normalized_validation
}

protected_financial_records = [
    record
    for record in finch_records
    if record["db_name"] == "bird"
    and record["db_id"] == "financial"
]

assert len(protected_financial_records) == 106


def build_fingerprint(
    record: dict,
    sql: str,
    split: str,
) -> dict:
    normalized_question = normalize_question(
        record["question"]
    )

    normalized_sql = canonical_sql(sql)

    return {
        "database_id": record["db_id"],
        "question_id": record["question_id"],
        "split": split,
        "question_hash": exact_hash(
            normalized_question
        ),
        "question_tokens":
            question_token_hashes(
                record["question"]
            ),
        "sql_hash": exact_hash(
            normalized_sql
        ),
        "sql_shape_hash":
            sql_shape_hash(sql),
    }


clean_fingerprints = []

for record in clean_records:
    key = (
        record["db_id"],
        record["question_id"],
    )

    result = normalized_result_by_key[key]

    clean_fingerprints.append(
        build_fingerprint(
            record=record,
            sql=result["compatible_sql"],
            split=result["split"],
        )
    )


# Protected fingerprints remain in memory only.
protected_fingerprints = [
    build_fingerprint(
        record=record,
        sql=record["SQL"],
        split="protected",
    )
    for record in protected_financial_records
]


protected_question_hashes = {
    item["question_hash"]
    for item in protected_fingerprints
}

protected_sql_hashes = {
    item["sql_hash"]
    for item in protected_fingerprints
}


leakage_flags = []

for candidate in clean_fingerprints:
    reasons = []

    if (
        candidate["question_hash"]
        in protected_question_hashes
    ):
        reasons.append(
            "exact_protected_question"
        )

    if (
        candidate["sql_hash"]
        in protected_sql_hashes
    ):
        reasons.append(
            "exact_protected_sql"
        )

    maximum_similarity = max(
        jaccard_similarity(
            candidate["question_tokens"],
            protected["question_tokens"],
        )
        for protected
        in protected_fingerprints
    )

    if (
        maximum_similarity
        >= NEAR_DUPLICATE_THRESHOLD
    ):
        reasons.append(
            "near_protected_question"
        )

    if reasons:
        leakage_flags.append({
            "database_id":
                candidate["database_id"],
            "question_id":
                candidate["question_id"],
            "split": candidate["split"],
            "reasons": reasons,
            "maximum_similarity":
                maximum_similarity,
        })


def duplicate_groups(
    fingerprints: list[dict],
    field: str,
) -> list[list[tuple[str, int]]]:
    grouped = defaultdict(list)

    for item in fingerprints:
        grouped[item[field]].append(
            (
                item["database_id"],
                item["question_id"],
            )
        )

    return [
        members
        for members in grouped.values()
        if len(members) > 1
    ]


exact_question_groups = duplicate_groups(
    clean_fingerprints,
    "question_hash",
)

exact_sql_groups = duplicate_groups(
    clean_fingerprints,
    "sql_hash",
)

sql_shape_groups = duplicate_groups(
    clean_fingerprints,
    "sql_shape_hash",
)


train_fingerprints = [
    item
    for item in clean_fingerprints
    if item["split"] == "train"
]

validation_fingerprints = [
    item
    for item in clean_fingerprints
    if item["split"] == "validation"
]

cross_split_near_duplicates = []

for training_item in train_fingerprints:
    for validation_item in (
        validation_fingerprints
    ):
        similarity = jaccard_similarity(
            training_item[
                "question_tokens"
            ],
            validation_item[
                "question_tokens"
            ],
        )

        if (
            similarity
            >= NEAR_DUPLICATE_THRESHOLD
        ):
            cross_split_near_duplicates.append({
                "train_key": (
                    training_item[
                        "database_id"
                    ],
                    training_item[
                        "question_id"
                    ],
                ),
                "validation_key": (
                    validation_item[
                        "database_id"
                    ],
                    validation_item[
                        "question_id"
                    ],
                ),
                "similarity": similarity,
            })


print("Clean candidates:", len(
    clean_fingerprints
))
print(
    "Protected leakage flags:",
    len(leakage_flags),
)
print(
    "Exact question duplicate groups:",
    len(exact_question_groups),
)
print(
    "Exact SQL duplicate groups:",
    len(exact_sql_groups),
)
print(
    "SQL structural-template groups:",
    len(sql_shape_groups),
)
print(
    "Cross-split near-question pairs:",
    len(cross_split_near_duplicates),
)

if leakage_flags:
    print()
    print(
        "Flagged candidate IDs "
        "(protected content not displayed):"
    )

    for flag in leakage_flags:
        print(flag)

if cross_split_near_duplicates:
    print()
    print(
        "Cross-split candidate pairs:"
    )

    for pair in (
        cross_split_near_duplicates[:20]
    ):
        print(pair)


In [ ]:
normalized_result_by_key = {
    (result["database_id"], result["question_id"]): result
    for result in normalized_validation
}

enriched_clean_records = []
for source_record in clean_records:
    key = (source_record["db_id"], source_record["question_id"])
    validation_record = normalized_result_by_key[key]
    enriched_clean_records.append({
        **validation_record,
        "question": source_record["question"],
    })


def identities_for_groups(groups):
    return [sorted(group) for group in groups]


assert len(enriched_clean_records) == 310
assert not leakage_flags
assert not cross_split_near_duplicates

print("Enriched clean records:", len(enriched_clean_records))
print("Protected leakage flags:", len(leakage_flags))
print("Exact-question groups:", identities_for_groups(exact_question_groups))
print("Exact-SQL groups:", identities_for_groups(exact_sql_groups))
print("Cross-split near-question pairs:", len(cross_split_near_duplicates))
print("SQL structural-template groups:", len(sql_shape_groups))


### Leakage and duplicate result

- Protected leakage flags: **0**
- Cross-split near-question pairs: **0**
- One exact question/SQL duplicate: keep `regional_sales/296`, remove `/312`
- Two different validation questions share one SQL target: keep both as legitimate paraphrases
- Structural-template groups are retained because repetition of useful behavior is the curriculum

**Rule:** deduplicate examples, not behaviors.


## 8. Audit semantics—not merely execution

A deterministic, risk-based sample covers every database and the behaviors most relevant to the adaptation hypothesis. For each example, inspect table/join choice, filters, requested projections, arithmetic, aggregation, and invented assumptions.


In [ ]:
import hashlib

from sqlglot import exp


source_by_identity = {
    (
        record["database_id"],
        int(record["question_id"]),
    ): record
    for record in enriched_clean_records
}


def analyze_sql_behaviors(sql):
    tree = sqlglot.parse_one(sql, read="sqlite")

    tables = {
        table.name.casefold()
        for table in tree.find_all(exp.Table)
    }

    outer_select = (
        tree
        if isinstance(tree, exp.Select)
        else tree.find(exp.Select)
    )

    projection_count = (
        len(outer_select.expressions)
        if outer_select is not None
        else 0
    )

    has_aggregation = any(
        True
        for _ in tree.find_all(exp.AggFunc)
    )

    has_case = tree.find(exp.Case) is not None

    arithmetic_types = (
        exp.Add,
        exp.Sub,
        exp.Mul,
        exp.Div,
        exp.Mod,
    )

    select_count = sum(
        1
        for _ in tree.find_all(exp.Select)
    )

    behaviors = set()

    if tree.find(exp.Join) is not None:
        behaviors.add("explicit_join")

    if len(tables) >= 3:
        behaviors.add("three_plus_tables")

    if has_aggregation:
        behaviors.add("aggregation")

    if any(
        isinstance(node, arithmetic_types)
        for node in tree.walk()
    ):
        behaviors.add("arithmetic")

    if has_aggregation and has_case:
        behaviors.add("conditional_aggregation")

    if select_count > 1:
        behaviors.add("subquery")

    if projection_count > 1:
        behaviors.add("multi_column_projection")

    if tree.find(exp.Group) is not None:
        behaviors.add("grouping")

    if (
        tree.find(exp.Order) is not None
        and tree.find(exp.Limit) is not None
    ):
        behaviors.add("ranking_or_extreme")

    return behaviors


def stable_rank(record):
    identity = (
        f"{record['database_id']}|"
        f"{record['question_id']}"
    )
    return hashlib.sha256(
        identity.encode("utf-8")
    ).hexdigest()


annotated_records = []

for record in enriched_clean_records:
    annotated_records.append({
        **record,
        "behaviors": analyze_sql_behaviors(
            record["compatible_sql"]
        ),
    })


selected = []
selected_identities = set()


def add_first_matching(label, predicate):
    candidates = sorted(
        (
            record
            for record in annotated_records
            if predicate(record)
            and (
                record["database_id"],
                int(record["question_id"]),
            ) not in selected_identities
        ),
        key=stable_rank,
    )

    if not candidates:
        print(f"WARNING: no unused candidate for {label}")
        return

    chosen = candidates[0]
    identity = (
        chosen["database_id"],
        int(chosen["question_id"]),
    )

    selected_identities.add(identity)
    selected.append({
        "audit_reason": label,
        **chosen,
    })


# Ensure every database is represented.
for database_id in sorted({record['database_id'] for record in enriched_clean_records}):
    add_first_matching(
        f"database:{database_id}",
        lambda record, database_id=database_id:
            record["database_id"] == database_id,
    )


# Oversample behaviors that matter to the adaptation hypothesis.
for behavior in (
    "explicit_join",
    "three_plus_tables",
    "aggregation",
    "arithmetic",
    "conditional_aggregation",
    "subquery",
    "multi_column_projection",
    "ranking_or_extreme",
):
    add_first_matching(
        f"behavior:{behavior}",
        lambda record, behavior=behavior:
            behavior in record["behaviors"],
    )


print("Manual audit sample:", len(selected))

for index, record in enumerate(selected, start=1):
    print("=" * 88)
    print(f"Audit item {index}/{len(selected)}")
    print("Selection reason:", record["audit_reason"])
    print("Database:", record["database_id"])
    print("Question ID:", record["question_id"])
    print("Split:", record["split"])
    print("Difficulty:", record["difficulty"])
    print("Behaviors:", sorted(record["behaviors"]))

    print("\nQuestion:")
    print(record["question"])

    print("\nTraining target:")
    print(record["compatible_sql"])


In [ ]:
import sqlite3


regional_sales_path = (
    repository_root
    / "data"
    / "training_databases"
    / "regional_sales"
    / "regional_sales.sqlite"
)


with sqlite3.connect(regional_sales_path) as connection:
    # Question 268: inspect integer-versus-real averaging.
    order_count = connection.execute(
        """
        SELECT COUNT(t1.ordernumber)
        FROM "sales orders" AS t1
        INNER JOIN customers AS t2
            ON t2.customerid = t1._customerid
        WHERE (
            t1.orderdate LIKE '%/%/18'
            AND t2."customer names" = 'weimei corp' COLLATE NOCASE
        )
        OR (
            t1.orderdate LIKE '%/%/19'
            AND t2."customer names" = 'weimei corp' COLLATE NOCASE
        )
        OR (
            t1.orderdate LIKE '%/%/20'
            AND t2."customer names" = 'weimei corp' COLLATE NOCASE
        )
        """
    ).fetchone()[0]

    sqlite_integer_average = connection.execute(
        "SELECT ? / 3",
        (order_count,),
    ).fetchone()[0]

    real_average = connection.execute(
        "SELECT CAST(? AS REAL) / 3",
        (order_count,),
    ).fetchone()[0]

    # Question 181: determine whether duplicate names occur.
    online_customer_names = [
        row[0]
        for row in connection.execute(
            """
            SELECT t1."customer names"
            FROM customers AS t1
            INNER JOIN "sales orders" AS t2
                ON t2._customerid = t1.customerid
            WHERE t2."sales channel" = 'online' COLLATE NOCASE
            """
        )
    ]


print("Question 268")
print("Order count:", order_count)
print("Current integer result:", sqlite_integer_average)
print("Real-valued result:", real_average)

print("\nQuestion 181")
print("Returned rows:", len(online_customer_names))
print("Unique customer names:", len(set(online_customer_names)))
print(
    "Duplicate rows:",
    len(online_customer_names) - len(set(online_customer_names)),
)


### Semantic audit result and decisions

| IDs | Decision | Evidence |
|---|---|---|
| 158, 419, 206, 165, 332, 409, 324 | Keep | Question and SQL align |
| 191 | Exclude | Does not determine the less-common type and projects type instead of region |
| 175 | Exclude | Projects product ID instead of requested order number |
| 268 | Correct | SQLite integer division returns 53 instead of 53.666... |
| 181 | Correct | Returns 2,425 rows for only 50 unique customer names |

The audit caught labels that parsed and executed successfully. That is the most important finding in this milestone: **technical validity is necessary, but semantic validity is the actual label-quality gate.**


In [ ]:
# Conservative, evidence-backed decisions from duplicate and semantic review.
CURATION_EXCLUSIONS = {
    ("regional_sales", 312): {
        "reason": "exact_duplicate",
        "evidence": "Same normalized question and SQL as regional_sales/296.",
    },
    ("regional_sales", 191): {
        "reason": "semantic_label_failure",
        "evidence": (
            "Question requests the less-common store type and region, but the "
            "reference neither compares counts nor projects region."
        ),
    },
    ("regional_sales", 175): {
        "reason": "semantic_label_failure",
        "evidence": (
            "Question requests order number and product name, but the reference "
            "projects product ID and product name."
        ),
    },
}

MANUAL_SQL_CORRECTIONS = {
    ("regional_sales", 268): {
        "reason": "prevent_integer_truncation",
        "evidence": "161 / 3 is 53 in SQLite integer division, not 53.666...",
        "sql": """SELECT CAST(COUNT(t1.ordernumber) AS REAL) / 3.0
FROM "sales orders" AS t1
INNER JOIN customers AS t2 ON t2.customerid = t1._customerid
WHERE (t1.orderdate LIKE '%/%/18' AND t2."customer names" = 'weimei corp' COLLATE NOCASE)
   OR (t1.orderdate LIKE '%/%/19' AND t2."customer names" = 'weimei corp' COLLATE NOCASE)
   OR (t1.orderdate LIKE '%/%/20' AND t2."customer names" = 'weimei corp' COLLATE NOCASE)""",
    },
    ("regional_sales", 181): {
        "reason": "deduplicate_requested_entities",
        "evidence": "Original returned 2,425 rows for only 50 unique customer names.",
        "sql": """SELECT DISTINCT t1."customer names"
FROM customers AS t1
INNER JOIN "sales orders" AS t2 ON t2._customerid = t1.customerid
WHERE t2."sales channel" = 'online' COLLATE NOCASE""",
    },
}


# Validate every hand correction structurally and against the real database.
for (database_id, question_id), correction in MANUAL_SQL_CORRECTIONS.items():
    structure = validate_sql_structure(correction["sql"])
    assert structure["parsed"]
    assert structure["single_statement"]
    assert structure["read_only"]

    sqlite_path = Path(asset_inventory[database_id]["sqlite_path"])
    with sqlite3.connect(
        f"file:{sqlite_path.as_posix()}?mode=ro",
        uri=True,
    ) as connection:
        execution = execute_with_timeout(connection, correction["sql"])

    status, concerns = classify_execution(structure, execution)
    assert status == "passed_technical_validation", (question_id, status, concerns)


final_records = []
for record in enriched_clean_records:
    identity = (record["database_id"], int(record["question_id"]))
    if identity in CURATION_EXCLUSIONS:
        continue

    final_record = dict(record)
    if identity in MANUAL_SQL_CORRECTIONS:
        correction = MANUAL_SQL_CORRECTIONS[identity]
        final_record["compatible_sql"] = correction["sql"].strip()
        final_record["manual_correction"] = {
            key: value
            for key, value in correction.items()
            if key != "sql"
        }
    final_records.append(final_record)

final_records.sort(
    key=lambda record: (
        record["split"],
        record["database_id"],
        int(record["question_id"]),
    )
)

split_counts = Counter(record["split"] for record in final_records)
database_counts = Counter(record["database_id"] for record in final_records)

assert len(final_records) == 307
assert split_counts == {"train": 222, "validation": 85}
assert all(record["database_id"] != "financial" for record in final_records)

final_pool_payload = {
    "artifact_version": "1.0",
    "purpose": "Leakage-safe, validated candidate pool for a controlled research SFT experiment.",
    "readiness": "research_sft_ready",
    "record_count": len(final_records),
    "split_counts": dict(sorted(split_counts.items())),
    "database_counts": dict(sorted(database_counts.items())),
    "curation_policy": {
        "protected_financial_records_allowed": False,
        "technical_failures_allowed": False,
        "suspicious_execution_results_allowed": False,
        "exact_duplicates": "keep_one",
        "semantic_label_failures": "exclude",
        "manual_corrections": "require written evidence plus parse and execution validation",
        "different_questions_same_sql": "retain as legitimate paraphrases",
    },
    "exclusions": [
        {"database_id": db, "question_id": qid, **decision}
        for (db, qid), decision in sorted(CURATION_EXCLUSIONS.items())
    ],
    "manual_corrections": [
        {
            "database_id": db,
            "question_id": qid,
            "reason": decision["reason"],
            "evidence": decision["evidence"],
        }
        for (db, qid), decision in sorted(MANUAL_SQL_CORRECTIONS.items())
    ],
    "limitations": [
        "Representative semantic sampling is not exhaustive human review.",
        "FINCH is CC-BY-NC-4.0 and is used only for research and education.",
        "Exact token counts require the tokenizer of the eventual fine-tuning model.",
    ],
    "records": final_records,
}

FINAL_POOL_PATH = TRAINING_DIR / "curated_candidate_pool_v1.json"
serialized_pool = json.dumps(
    final_pool_payload,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
) + "\n"
FINAL_POOL_PATH.write_text(serialized_pool, encoding="utf-8")
final_pool_sha256 = sha256(serialized_pool.encode("utf-8")).hexdigest()

print("Final records:", len(final_records))
print("By split:", dict(split_counts))
print("By database:", dict(database_counts))
print("Excluded:", sorted(CURATION_EXCLUSIONS))
print("Corrected:", sorted(MANUAL_SQL_CORRECTIONS))
print("Saved:", FINAL_POOL_PATH)
print("Final-pool SHA-256:", final_pool_sha256)


## 9. Match the SFT record to production inference

Each demonstration has exactly three messages:

| Role | Content |
|---|---|
| `system` | The frozen Milestone 5 SQL-construction instructions |
| `user` | Relationship-aware schema context plus the question, using the production template |
| `assistant` | Validated SQL only—no rationale, Markdown, or hidden chain of thought |

Metadata stays in the canonical audit artifact. Provider JSONL contains only `messages`, because fine-tuning APIs may reject extra fields.


In [ ]:
import json
import sqlite3
from pathlib import Path

from domain_adapt.frontier import build_model_input


sft_source_records = final_records


# Reuse the exact Milestone 5 production instructions.
config_path = (
    repository_root
    / "configs"
    / "slm_context_engineered_v1.json"
)

config = json.loads(config_path.read_text(encoding="utf-8"))
training_instructions = config["instructions"]


def quote_identifier(identifier):
    return '"' + str(identifier).replace('"', '""') + '"'


def build_relationship_schema_context(sqlite_path):
    """Build compact schema context with PK and FK relationships."""

    with sqlite3.connect(sqlite_path) as connection:
        tables = [
            row[0]
            for row in connection.execute(
                """
                SELECT name
                FROM sqlite_master
                WHERE type = 'table'
                  AND name NOT LIKE 'sqlite_%'
                ORDER BY name
                """
            )
        ]

        schema_lines = []

        for table in tables:
            columns = connection.execute(
                f"PRAGMA table_info({quote_identifier(table)})"
            ).fetchall()

            foreign_keys = connection.execute(
                f"PRAGMA foreign_key_list({quote_identifier(table)})"
            ).fetchall()

            foreign_key_targets = {
                row[3]: (row[2], row[4])
                for row in foreign_keys
            }

            column_descriptions = []

            for column in columns:
                column_name = column[1]
                column_type = column[2] or "UNKNOWN"

                labels = []

                if column[5]:
                    labels.append("PK")

                if column_name in foreign_key_targets:
                    target_table, target_column = (
                        foreign_key_targets[column_name]
                    )
                    labels.append(
                        f"FK->{target_table}.{target_column}"
                    )

                suffix = (
                    f" {' '.join(labels)}"
                    if labels
                    else ""
                )

                column_descriptions.append(
                    f"{quote_identifier(column_name)} "
                    f"{column_type}{suffix}"
                )

            schema_lines.append(
                f"- {quote_identifier(table)}: "
                + "; ".join(column_descriptions)
                + "."
            )

    return (
        "SQLite database.\n\n"
        "Schema:\n"
        + "\n".join(schema_lines)
    )


# Pick a representative training example containing a JOIN.
sample = next(
    record
    for record in sft_source_records
    if record["split"] == "train"
    and " join " in record["compatible_sql"].casefold()
)

database_id = sample["database_id"]

sqlite_path = (
    repository_root
    / "data"
    / "training_databases"
    / database_id
    / f"{database_id}.sqlite"
)

assert sqlite_path.exists(), sqlite_path

schema_context = build_relationship_schema_context(
    sqlite_path
)

user_message = build_model_input(
    schema_context,
    sample["question"],
)

canonical_example = {
    "messages": [
        {
            "role": "system",
            "content": training_instructions,
        },
        {
            "role": "user",
            "content": user_message,
        },
        {
            "role": "assistant",
            "content": sample["compatible_sql"].strip(),
        },
    ],
    "metadata": {
        "database_id": database_id,
        "question_id": sample["question_id"],
        "split": sample["split"],
        "difficulty": sample["difficulty"],
        "reference_changed": sample["reference_changed"],
        "compatibility_rules": sample["compatibility_rules"],
    },
}


# Contract safeguards.
assert [message["role"] for message in canonical_example["messages"]] == [
    "system",
    "user",
    "assistant",
]

assert canonical_example["messages"][0]["content"] == config["instructions"]
assert sample["question"] in canonical_example["messages"][1]["content"]
assert canonical_example["messages"][2]["content"]
assert "```" not in canonical_example["messages"][2]["content"]
assert database_id != "financial"


print("Database:", database_id)
print("Question ID:", sample["question_id"])
print("Difficulty:", sample["difficulty"])
print("System characters:", len(training_instructions))
print("Schema characters:", len(schema_context))
print("User-message characters:", len(user_message))

print("\nQuestion:")
print(sample["question"])

print("\nAssistant training target:")
print(canonical_example["messages"][2]["content"])

print("\nCanonical SFT example validated.")
print("No model request made.")


## 10. Export versioned artifacts

The final generation step writes:

- `curated_candidate_pool_v1.json` — curation evidence and source records;
- `canonical_sft_records_v1.json` — messages plus provenance;
- `sft_train_v1.jsonl` — provider-ready training messages;
- `sft_validation_v1.jsonl` — held-out-schema validation messages;
- `sft_manifest_v1.json` — counts, hashes, context versions, and limitations.

Character counts are a provider-neutral size proxy. Exact token counts must be computed with the tokenizer of the model selected in Milestone 8; inventing a tokenizer now would give false precision.


In [ ]:
import hashlib
import json
import math
import statistics
from collections import Counter
from pathlib import Path

import sqlglot


training_dir = repository_root / "data" / "training"
training_dir.mkdir(parents=True, exist_ok=True)


def sha256_bytes(content):
    return hashlib.sha256(content).hexdigest()


def sha256_file(path):
    return sha256_bytes(path.read_bytes())


def percentile(values, percentage):
    ordered = sorted(values)
    index = math.ceil(percentage * len(ordered)) - 1
    return ordered[max(0, index)]


# Build each database context only once.
database_ids = sorted({
    record["database_id"]
    for record in final_records
})

schema_contexts = {}

for database_id in database_ids:
    sqlite_path = (
        repository_root
        / "data"
        / "training_databases"
        / database_id
        / f"{database_id}.sqlite"
    )

    assert sqlite_path.exists(), sqlite_path

    schema_contexts[database_id] = (
        build_relationship_schema_context(sqlite_path)
    )


canonical_records = []

for record in final_records:
    database_id = record["database_id"]
    question = record["question"].strip()
    target = record["compatible_sql"].strip()

    # Validate that the target is parseable SQLite SQL.
    parsed_target = sqlglot.parse_one(
        target,
        read="sqlite",
    )

    assert parsed_target is not None
    assert question
    assert target
    assert "```" not in target
    assert database_id != "financial"

    messages = [
        {
            "role": "system",
            "content": training_instructions,
        },
        {
            "role": "user",
            "content": build_model_input(
                schema_contexts[database_id],
                question,
            ),
        },
        {
            "role": "assistant",
            "content": target,
        },
    ]

    canonical_records.append({
        "messages": messages,
        "metadata": {
            "database_id": database_id,
            "question_id": record["question_id"],
            "split": record["split"],
            "difficulty": record["difficulty"],
            "reference_changed": record["reference_changed"],
            "compatibility_rules": record["compatibility_rules"],
            "manual_correction": record.get("manual_correction"),
        },
    })


# Stable ordering supports reproducible hashes.
canonical_records.sort(
    key=lambda item: (
        item["metadata"]["split"],
        item["metadata"]["database_id"],
        int(item["metadata"]["question_id"]),
    )
)


train_records = [
    record
    for record in canonical_records
    if record["metadata"]["split"] == "train"
]

validation_records = [
    record
    for record in canonical_records
    if record["metadata"]["split"] == "validation"
]


# Final contract checks.
assert len(canonical_records) == 307
assert len(train_records) == 222
assert len(validation_records) == 85

assert len({
    (
        record["metadata"]["database_id"],
        record["metadata"]["question_id"],
    )
    for record in canonical_records
}) == 307

assert all(
    [message["role"] for message in record["messages"]]
    == ["system", "user", "assistant"]
    for record in canonical_records
)

assert all(
    record["messages"][0]["content"]
    == training_instructions
    for record in canonical_records
)


canonical_path = (
    training_dir / "canonical_sft_records_v1.json"
)

train_jsonl_path = (
    training_dir / "sft_train_v1.jsonl"
)

validation_jsonl_path = (
    training_dir / "sft_validation_v1.jsonl"
)


canonical_payload = {
    "artifact_version": "1.0",
    "record_count": len(canonical_records),
    "records": canonical_records,
}

canonical_path.write_text(
    json.dumps(
        canonical_payload,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


def write_provider_jsonl(path, canonical_subset):
    lines = [
        json.dumps(
            {"messages": record["messages"]},
            ensure_ascii=False,
            sort_keys=True,
        )
        for record in canonical_subset
    ]

    path.write_text(
        "\n".join(lines) + "\n",
        encoding="utf-8",
    )


write_provider_jsonl(
    train_jsonl_path,
    train_records,
)

write_provider_jsonl(
    validation_jsonl_path,
    validation_records,
)


# Character counts are a provider-neutral size estimate.
prompt_characters = [
    len(record["messages"][0]["content"])
    + len(record["messages"][1]["content"])
    for record in canonical_records
]

target_characters = [
    len(record["messages"][2]["content"])
    for record in canonical_records
]

# Provider-neutral approximation only; use the selected model's tokenizer in M8.
prompt_token_estimates = [math.ceil(value / 4) for value in prompt_characters]
target_token_estimates = [math.ceil(value / 4) for value in target_characters]


manifest = {
    "artifact_version": "1.0",
    "source_candidate_pool": {
        "path": "data/training/curated_candidate_pool_v1.json",
        "sha256": sha256_file(FINAL_POOL_PATH),
    },
    "readiness": "research_sft_ready",
    "semantic_curation": {
        "excluded": len(CURATION_EXCLUSIONS),
        "corrected": len(MANUAL_SQL_CORRECTIONS),
    },
    "record_counts": {
        "total": len(canonical_records),
        "train": len(train_records),
        "validation": len(validation_records),
    },
    "database_counts": dict(sorted(Counter(
        record["metadata"]["database_id"]
        for record in canonical_records
    ).items())),
    "schema_context_sha256": {
        database_id: sha256_bytes(
            context.encode("utf-8")
        )
        for database_id, context
        in sorted(schema_contexts.items())
    },
    "instructions_sha256": sha256_bytes(
        training_instructions.encode("utf-8")
    ),
    "artifacts": {
        "canonical": {
            "path": "data/training/canonical_sft_records_v1.json",
            "sha256": sha256_file(canonical_path),
        },
        "train_jsonl": {
            "path": "data/training/sft_train_v1.jsonl",
            "sha256": sha256_file(train_jsonl_path),
        },
        "validation_jsonl": {
            "path": "data/training/sft_validation_v1.jsonl",
            "sha256": sha256_file(validation_jsonl_path),
        },
    },
    "size_summary": {
        "prompt_characters_median": statistics.median(
            prompt_characters
        ),
        "prompt_characters_p95": percentile(
            prompt_characters,
            0.95,
        ),
        "target_characters_median": statistics.median(
            target_characters
        ),
        "target_characters_p95": percentile(
            target_characters,
            0.95,
        ),
        "approximate_prompt_tokens_median": statistics.median(prompt_token_estimates),
        "approximate_prompt_tokens_p95": percentile(prompt_token_estimates, 0.95),
        "approximate_target_tokens_median": statistics.median(target_token_estimates),
        "approximate_target_tokens_p95": percentile(target_token_estimates, 0.95),
    },
}

manifest_path = (
    training_dir / "sft_manifest_v1.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


print("Canonical records:", len(canonical_records))
print("Training records:", len(train_records))
print("Validation records:", len(validation_records))
print("Database counts:", manifest["database_counts"])

print("\nPrompt characters:")
print("  Median:", manifest["size_summary"]["prompt_characters_median"])
print("  P95:", manifest["size_summary"]["prompt_characters_p95"])

print("Target characters:")
print("  Median:", manifest["size_summary"]["target_characters_median"])
print("  P95:", manifest["size_summary"]["target_characters_p95"])

print("\nArtifacts:")
for name, details in manifest["artifacts"].items():
    print(f"  {name}: {details['sha256']}")

print("  manifest:", sha256_file(manifest_path))
print("\nNo model request made.")


## Milestone 7 conclusion

The expected final package contains **307 examples**: **222 training** and **85 validation**. It excludes all 106 protected `financial` records, 25 technically suspicious/failing records, one exact duplicate, and two semantic label failures; two additional labels receive evidence-backed corrections.

### Readiness decision

**GO for a controlled research SFT experiment**, subject to these boundaries:

- this is not production-grade or commercially licensed training data;
- the semantic audit is representative rather than exhaustive;
- the complete frozen `financial` evaluation remains untouched;
- Milestone 8 must record the exact base model, tokenizer, hyperparameters, Azure job, and generated artifact hashes;
- success is determined only by post-SFT frozen evaluation—not training loss alone.

The reusable lesson is simple: **freeze evaluation first, validate assets and labels independently, teach the same contract used at inference, and make every exclusion or correction auditable.**
